# Clase 1 · El sancocho

**Taller en Ciencia de Datos y Aprendizaje Automático** · Especialización en Estadística · UNAL Manizales · 2026-II
Profesor: Carlos Manuel Orrego Franco

Este cuaderno lo resuelvo yo en vivo durante la primera mitad de la clase. Ustedes lo siguen, lo corren y me interrumpen. En la segunda mitad cada uno resuelve el gemelo, *El precio de la luz*, con la misma técnica y otros datos.

| Ficha | |
|---|---|
| **Objetivo pedagógico** | Convertir una pregunta de la vida diaria en un problema de datos con métrica, explorarlo, comparar cinco modelos contra una línea base y validar respetando el tiempo. |
| **Enfoque** | Ciencia de Datos. |
| **Resultados de aprendizaje** | 1 (formular el problema, los objetivos y la métrica), 3 (flujo reproducible y trazable), 4 (implementar y evaluar un modelo) y 5 (comunicar a quien no es técnico). |
| **Uso de IA** | Pueden pedirle a Claude que explique una línea de código o que escriba una gráfica. Deben poder explicar sin ayuda qué es una línea base, por qué no se barajan los meses, por qué el ganador se escoge en validación y qué significa el error que reportan. |
| **Entregable** | Este cuaderno corrido de principio a fin en su portafolio, con la decisión de la sección 4 escrita por ustedes. |

In [ ]:
# Celda de arranque. No hay que cambiarle nada: trae las herramientas del curso.
import pathlib, subprocess, sys
raiz = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "taller" / "datos.py").exists()), None)
if raiz is None:
    subprocess.run(["git", "clone", "-q", "https://github.com/cmorregof/taller-cd-ml-2026-2.git"], check=True)
    raiz = pathlib.Path("taller-cd-ml-2026-2")
sys.path.insert(0, str(raiz))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from taller import descargar, procedencia

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False, "axes.spines.right": False})
print("Herramientas listas.")

## 1. Formulación

Imagínense a alguien que vive de vender sancocho en Manizales. Compra en la Galería, cocina para seis por olla y tiene que imprimir la carta con el precio del plato. La carta dura un trimestre. Si el mercado sube y el precio se quedó quieto, el trimestre se lo come el espinazo.

Esa persona no necesita un modelo. Necesita tres respuestas:

1. **Predecir.** ¿Cuánto costará la olla dentro de tres meses?
2. **Explicar.** ¿Qué ingrediente mueve el costo?
3. **Decidir.** ¿Con qué costo debe presupuestar la olla del próximo trimestre para no quedar corta?

**El dato.** El DANE publica cada mes, en el sistema SIPSA, el precio mayorista por kilo de cientos de productos en las centrales de abasto del país. Manizales reporta por la Galería.

**La receta, para seis:** 1,5 kg de espinazo de cerdo, 1 kg de yuca, 1 kg de papa, tres mazorcas (0,9 kg) y cuatro plátanos verdes (1,4 kg).

**La métrica.** Medimos el error porcentual absoluto medio (MAPE) del pronóstico a tres meses. Un MAPE de 6 % significa que, en promedio, el pronóstico se aleja 6 % del costo real. Elegimos un porcentaje porque el dueño del negocio piensa en márgenes, no en pesos sueltos.

### Predicción escrita

Antes de ver un solo dato, escriban aquí su apuesta. Después la comparamos con lo que salga.

- ¿Cuánto costaban los ingredientes de la olla en diciembre de 2024, al por mayor? **Mi apuesta:**
- ¿Qué ingrediente pesa más en el costo? **Mi apuesta:**
- Si yo pronostico «dentro de tres meses costará lo mismo que hoy», ¿de qué tamaño será mi error? **Mi apuesta:**

## 2. Datos

Una sola línea trae el dato desde el DANE. La función descarga los ocho archivos oficiales, los unifica y revisa con una huella (MD5) que el resultado sea idéntico al que usé para preparar la clase. Tarda cerca de medio minuto.

El archivo no está guardado en el repositorio del curso porque el DANE no permite republicarlo. Por eso cada uno lo descarga de la fuente.

In [ ]:
sipsa = descargar("sipsa")
sipsa.head()

In [ ]:
print("Filas:", len(sipsa))
print("Meses:", sipsa["fecha"].min(), "a", sipsa["fecha"].max())
print("Ciudades:", sipsa["ciudad"].nunique(), "| Productos:", sipsa["producto"].nunique())

Nos quedamos con Manizales. Antes de filtrar productos conviene mirar cómo se llaman, porque el DANE no dice «papa»: dice la variedad.

In [ ]:
manizales = sipsa[sipsa["ciudad"] == "Manizales"]
print(manizales["mercado"].unique())
for palabra in ["espinazo", "Yuca", "Papa", "mazorca", "Plátano"]:
    print(palabra, "->", sorted(p for p in manizales["producto"].unique() if palabra.lower() in p.lower()))

Hay varias papas y varios plátanos. Escoger uno es una **decisión de modelación** y va a la bitácora de la sección 4. Tomo las variedades que se venden todos los meses en la Galería.

In [ ]:
receta = {                                  # kilos para seis personas
    "Carne de cerdo, espinazo": 1.5,
    "Yuca ICA": 1.0,
    "Papa capira": 1.0,
    "Chócolo mazorca": 0.9,
    "Plátano dominico hartón verde": 1.4,
}
precios = (manizales[manizales["producto"].isin(receta)]
           .pivot_table(index="fecha", columns="producto", values="precio_kg"))
precios.index = pd.to_datetime(precios.index)
print("Meses:", len(precios), "| Faltantes por producto:")
print(precios.isna().sum())

Ningún mes falta. El costo de la olla es la suma de precio por cantidad.

In [ ]:
costo_por_ingrediente = precios * pd.Series(receta)
olla = costo_por_ingrediente.sum(axis=1).rename("olla")

ax = olla.plot(color="#A8432C", lw=2)
ax.set(title="Costo mayorista de una olla de sancocho para seis, Manizales", xlabel="", ylabel="pesos")
plt.show()
print(f"Enero de 2013: ${olla.iloc[0]:,.0f} | Diciembre de 2024: ${olla.iloc[-1]:,.0f} | Se multiplicó por {olla.iloc[-1] / olla.iloc[0]:.2f}")

Comparen con su apuesta. Ojo con lo que mide esta cifra: es el costo **mayorista** de los ingredientes en pesos corrientes. No incluye gas, mano de obra ni el margen de la tienda.

### 2.1 Conocer el dato antes de modelarlo

Antes de pronosticar nada conviene mirar el dato por cuatro lados: cuánto vale cada ingrediente, cuánto se mueve de un mes a otro, si tiene temporada y cuánta memoria tiene. Llamo $p_{i,t}$ al precio por kilo del ingrediente $i$ en el mes $t$ y $q_i$ a los kilos de la receta. El costo de la olla es

$$C_t=\sum_{i=1}^{5} q_i\,p_{i,t},$$

y la variación mensual de cada precio es

$$r_{i,t}=\frac{p_{i,t}-p_{i,t-1}}{p_{i,t-1}}.$$

In [ ]:
display(precios.describe().round(0))

ejes = precios.plot(subplots=True, layout=(2, 3), figsize=(12, 5), legend=False, color="#16302B", lw=1.2)
for eje, nombre in zip(ejes.ravel(), precios.columns):
    eje.set_title(nombre, fontsize=9); eje.set_xlabel("")
ejes.ravel()[-1].axis("off")
plt.suptitle("Precio mayorista por kilo, Manizales"); plt.tight_layout(); plt.show()

In [ ]:
cambios = precios.pct_change().dropna()            # variación mensual de cada ingrediente

fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 4))
(cambios * 100).plot.box(ax=izq, rot=20, color="#16302B")
izq.set(title="Variación mensual por ingrediente", ylabel="%"); izq.tick_params(axis="x", labelsize=7)

correlacion = cambios.corr()
imagen = der.imshow(correlacion, cmap="RdBu_r", vmin=-1, vmax=1)
der.set_xticks(range(5)); der.set_yticks(range(5))
der.set_xticklabels([c[:12] for c in correlacion.columns], rotation=30, fontsize=7); der.set_yticklabels([c[:12] for c in correlacion.columns], fontsize=7)
for a in range(5):
    for b in range(5):
        der.text(b, a, f"{correlacion.iloc[a, b]:.2f}", ha="center", va="center", fontsize=8)
der.set_title("Correlación entre variaciones mensuales"); plt.colorbar(imagen, ax=der); plt.tight_layout(); plt.show()

print("Desviación estándar de la variación mensual (%):")
print((cambios.std() * 100).round(1).sort_values(ascending=False))

Dos lecturas. La primera es que hay ingredientes tranquilos e ingredientes nerviosos, y el más nervioso no tiene por qué ser el que más pesa en la olla. La segunda es que las variaciones casi no están correlacionadas entre sí: cada ingrediente tiene su propia historia. Eso lo retomamos con el PCA.

**Temporada y memoria.** Para ver la temporada comparo cada mes con el promedio móvil de doce meses centrado en él. Para ver la memoria uso la autocorrelación de orden $k$:

$$\hat\rho_k=\frac{\sum_{t=k+1}^{T}(C_t-\bar C)(C_{t-k}-\bar C)}{\sum_{t=1}^{T}(C_t-\bar C)^2}.$$

In [ ]:
tendencia = olla.rolling(12, center=True).mean()
indice_estacional = (olla / tendencia).groupby(olla.index.month).mean()

fig, (izq, centro, der) = plt.subplots(1, 3, figsize=(13, 3.6))
((indice_estacional - 1) * 100).plot.bar(ax=izq, color="#E0A526")
izq.set(title="Temporada: desvío frente a la tendencia", xlabel="mes", ylabel="%")

rezagos = range(1, 25)
centro.bar(rezagos, [olla.autocorr(k) for k in rezagos], color="#16302B")
centro.set(title="Autocorrelación del costo", xlabel="rezago (meses)")
der.bar(rezagos, [olla.pct_change().autocorr(k) for k in rezagos], color="#A8432C")
der.axhline(2 / np.sqrt(len(olla)), color="gray", ls=":"); der.axhline(-2 / np.sqrt(len(olla)), color="gray", ls=":")
der.set(title="Autocorrelación de la variación mensual", xlabel="rezago (meses)")
plt.tight_layout(); plt.show()

La gráfica del centro dice que el costo de hoy se parece mucho al de los meses anteriores: la serie tiene memoria larga porque tiene tendencia. La de la derecha quita esa tendencia y pregunta algo más fino: si este mes subió, ¿el próximo también? Las líneas punteadas marcan la banda $\pm 2/\sqrt{T}$, dentro de la cual una autocorrelación no se distingue del ruido.

Guarden esta imagen. Las variaciones tienen poca memoria: solo el primer rezago se asoma con claridad por fuera de la banda. Una serie cuyas variaciones no tienen memoria se llama **caminata aleatoria**, $C_{t+1}=C_t+\varepsilon_{t+1}$, y para una caminata aleatoria el mejor pronóstico es el último valor. La olla se le parece bastante, y eso anticipa lo que veremos en la sección 3.

### 2.2 PCA: ¿los ingredientes se mueven juntos?

El análisis de componentes principales busca las direcciones en que más varían los datos. Tomo la matriz $Z$ de variaciones mensuales estandarizadas, con una columna por ingrediente, y descompongo su matriz de correlaciones:

$$R=\frac{1}{T-1}Z^\top Z=V\Lambda V^\top,\qquad \Lambda=\operatorname{diag}(\lambda_1\ge\dots\ge\lambda_5).$$

Las columnas de $V$ son los componentes y $\lambda_j/\sum_k\lambda_k$ es la fracción de la varianza que explica el componente $j$. Si los cinco ingredientes obedecieran a una sola fuerza común, el primer componente explicaría casi todo. Si cada uno va por su lado, la varianza se reparte casi por igual, cerca de $1/5$ por componente.

**Predicción escrita:** ¿cuánto explicará el primer componente? **Mi apuesta:**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

Z = StandardScaler().fit_transform(cambios)
pca = PCA().fit(Z)
explicada = pca.explained_variance_ratio_ * 100
cargas = pd.DataFrame(pca.components_.T, index=cambios.columns, columns=[f"CP{j + 1}" for j in range(5)])

fig, (izq, der) = plt.subplots(1, 2, figsize=(12, 4))
izq.bar(cargas.columns, explicada, color="#16302B"); izq.axhline(20, color="gray", ls=":")
izq.set(title="Varianza explicada por componente", ylabel="%")
for nombre, (x, y) in cargas[["CP1", "CP2"]].iterrows():
    der.arrow(0, 0, x, y, color="#A8432C", head_width=0.02); der.text(x * 1.08, y * 1.08, nombre[:16], fontsize=8)
der.axhline(0, color="gray", lw=0.5); der.axvline(0, color="gray", lw=0.5)
der.set(title="Cargas de los dos primeros componentes", xlabel="CP1", ylabel="CP2", xlim=(-1, 1), ylim=(-1, 1))
plt.tight_layout(); plt.show()

print("Varianza explicada (%):", explicada.round(1))
display(cargas.round(2))

Lean la gráfica de la izquierda contra la línea punteada de 20 %, que es lo que explicaría cada componente si los ingredientes fueran independientes. En la de la derecha, dos flechas que apuntan al mismo lado son ingredientes que suben y bajan juntos, y dos flechas en ángulo recto son ingredientes que no tienen que ver.

El PCA aquí sirve para describir. No lo vamos a usar como entrada del modelo.

## 3. Modelado

### 3.1 La línea base

Todo modelo se mide contra el pronóstico más barato posible. Con horizonte $h=3$ meses hay dos candidatos que no cuestan nada:

$$\hat C^{\text{ing}}_{t+h}=C_t\qquad\text{y}\qquad \hat C^{\text{est}}_{t+h}=C_{t+h-12}.$$

El primero dice que dentro de tres meses costará lo mismo que hoy. El segundo, que costará lo mismo que ese mes del año pasado. El error se mide con

$$\text{MAPE}=\frac{100}{n}\sum_{t}\left|\frac{C_{t+h}-\hat C_{t+h}}{C_{t+h}}\right|.$$

Si un modelo sofisticado no les gana, no sirve, por elegante que sea.

In [ ]:
H = 3                                             # horizonte: tres meses
tabla = pd.DataFrame({"real": olla.shift(-H)})    # lo que queremos adivinar: el costo en t+3
tabla["ingenuo"] = olla                           # lo que sabemos hoy
tabla["estacional"] = olla.shift(12 - H)          # el mismo mes objetivo, un año antes
tabla = tabla.dropna()

def mape(real, pronostico):
    return float((abs(real - pronostico) / real).mean() * 100)

for metodo in ["ingenuo", "estacional"]:
    print(f"MAPE {metodo}: {mape(tabla['real'], tabla[metodo]):.1f} %")

Ya hay una cifra que vencer. Comparen con su tercera apuesta.

### 3.2 La trampa de barajar

Empecemos por el modelo más simple, una regresión lineal sobre el costo de hoy, el de los dos meses anteriores y el del mismo mes objetivo un año atrás:

$$C_{t+h}=\beta_0+\beta_1C_t+\beta_2C_{t-1}+\beta_3C_{t-2}+\beta_4C_{t+h-12}+\varepsilon_{t+h}.$$

In [ ]:
X = pd.DataFrame({
    "hoy": olla,
    "hace_1": olla.shift(1),
    "hace_2": olla.shift(2),
    "anio_pasado": olla.shift(12 - H),
})
datos = X.join(olla.shift(-H).rename("real")).dropna()
variables = list(X.columns)
print(len(datos), "meses utilizables, de", datos.index.min().date(), "a", datos.index.max().date())

Ahora viene la decisión que más errores causa en la práctica: **cómo separar entrenamiento y prueba**. Hagámoslo de las dos maneras y comparemos.

- **Barajando:** se toma al azar el 25 % de los meses para probar. Es lo que hace `train_test_split` por defecto.
- **Respetando el tiempo:** se entrena con todo hasta diciembre de 2021 y se prueba con 2022 en adelante.

**Predicción escrita:** ¿cuál de las dos dará un error más pequeño, y por qué? **Mi apuesta:**

In [ ]:
from sklearn.model_selection import train_test_split

# Manera 1: barajando. Se repite 200 veces para que el resultado no dependa de un sorteo con suerte.
barajados = []
for semilla in range(200):
    X_a, X_b, y_a, y_b = train_test_split(datos[variables], datos["real"], test_size=0.25, random_state=semilla)
    barajados.append(mape(y_b, LinearRegression().fit(X_a, y_a).predict(X_b)))
mape_barajado = float(np.mean(barajados))

# Manera 2: respetando el tiempo
antes, despues = datos.loc[:"2021-12-01"], datos.loc["2022-01-01":]
mape_temporal = mape(despues["real"], LinearRegression().fit(antes[variables], antes["real"]).predict(despues[variables]))

print(f"MAPE barajando (promedio de 200 sorteos): {mape_barajado:.1f} %")
print(f"MAPE respetando el tiempo:                 {mape_temporal:.1f} %")

Los dos números miden el mismo modelo, así que la diferencia viene del examen. Al barajar, el modelo se entrena con meses de 2023 y lo examinan con meses de 2022: ya vio el futuro. Ese examen no se parece a lo que le pasará al dueño del negocio, que siempre pronostica hacia adelante. De aquí en adelante solo vale la validación que respeta el tiempo.

### 3.3 Cinco modelos y un protocolo para escoger

Una regresión es un candidato entre muchos. Vamos a poner a competir cinco familias:

| Modelo | Idea en una línea |
|---|---|
| Regresión lineal | $\hat y=\beta_0+x^\top\beta$, con $\beta$ que minimiza $\sum(y-\hat y)^2$ |
| Ridge | Lo mismo, con castigo a los coeficientes grandes: $\min_\beta\sum(y-\hat y)^2+\alpha\lVert\beta\rVert_2^2$ |
| $k$ vecinos | Promedia el resultado de los $k$ meses pasados más parecidos al de hoy |
| Bosque aleatorio | Promedia muchos árboles de decisión entrenados sobre remuestreos del dato |
| *Gradient boosting* | Suma árboles pequeños, cada uno corrigiendo el error del anterior |

Si escogemos al ganador mirando la prueba, la prueba deja de ser un examen limpio: de cinco candidatos, alguno saldrá bien por suerte. Por eso el tiempo se parte en **tres** tramos:

$$\underbrace{2013\ \dots\ 2019}_{\text{entrenamiento}}\ \Big|\ \underbrace{2020\ \dots\ 2021}_{\text{validación: se escoge}}\ \Big|\ \underbrace{2022\ \dots\ 2024}_{\text{prueba: se reporta}}$$

Hay un detalle fino. La fila del mes $t$ lleva como respuesta el costo de $t+h$. Para que el entrenamiento no use respuestas que caen dentro del tramo siguiente, solo entran las filas con $t+h$ anterior al corte.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def cinco_modelos():
    return {
        "lineal": LinearRegression(),
        "ridge": make_pipeline(StandardScaler(), Ridge(alpha=10)),
        "k vecinos": make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
        "bosque": RandomForestRegressor(n_estimators=300, random_state=0),
        "boosting": GradientBoostingRegressor(random_state=0),
    }

objetivo = datos.index + pd.DateOffset(months=H)                 # el mes que cada fila intenta adivinar
E = objetivo <= pd.Timestamp("2019-12-01")                        # entrenamiento
V = (datos.index >= "2020-01-01") & (objetivo <= pd.Timestamp("2021-12-01"))   # validación
P = datos.index >= "2022-01-01"                                   # prueba
print(f"Entrenamiento: {E.sum()} meses | Validación: {V.sum()} | Prueba: {P.sum()}")

def competir(entradas, respuesta, a_pesos):
    """Mide cada modelo en validación; luego lo reentrena con entrenamiento y validación y lo mide en prueba."""
    errores, pronosticos = {}, {}
    for nombre, modelo in cinco_modelos().items():
        modelo.fit(entradas[E], respuesta[E])
        en_validacion = mape(datos["real"][V], a_pesos(modelo.predict(entradas[V]), V))
        modelo.fit(entradas[E | V], respuesta[E | V])
        pronosticos[nombre] = pd.Series(a_pesos(modelo.predict(entradas[P]), P), index=datos.index[P])
        errores[nombre] = {"validación": en_validacion, "prueba": mape(datos["real"][P], pronosticos[nombre])}
    return pd.DataFrame(errores).T, pd.DataFrame(pronosticos)

errores_nivel, pron_nivel = competir(datos[variables], datos["real"], lambda pred, tramo: pred)
bases = pd.DataFrame({m: {"validación": mape(datos["real"][V], tabla.loc[datos.index[V], m]),
                          "prueba": mape(datos["real"][P], tabla.loc[datos.index[P], m])} for m in ["ingenuo", "estacional"]}).T
display(pd.concat([bases, errores_nivel]).round(1))

In [ ]:
fechas = datos.index[P] + pd.DateOffset(months=H)
plt.figure(figsize=(11, 4))
plt.plot(fechas, datos["real"][P], color="black", lw=2.5, label="real")
plt.plot(fechas, tabla.loc[datos.index[P], "ingenuo"], color="#E0A526", lw=1.5, label="ingenuo")
for nombre in pron_nivel:
    plt.plot(fechas, pron_nivel[nombre], lw=1, ls="--", label=nombre)
plt.axhline(datos["real"][E | V].max(), color="gray", ls=":")
plt.text(fechas[0], datos["real"][E | V].max() * 1.005, "máximo visto antes de 2022", fontsize=8, color="gray")
plt.title("Pronóstico a tres meses en la prueba"); plt.ylabel("pesos"); plt.legend(fontsize=8, ncol=4); plt.show()

Miren las líneas del bosque, del *boosting* y de los vecinos contra la línea punteada. Un árbol predice promediando respuestas que ya vio, y un vecino también. Ninguno puede pronosticar un costo más alto que el máximo de su entrenamiento. La olla de 2022 a 2024 cuesta más que cualquier olla anterior, así que esos modelos se quedan pegados al techo. Los modelos lineales sí extrapolan.

Esto tiene arreglo, y el arreglo es una decisión de formulación. En vez de pedirles el **nivel** del costo, les pedimos el **cambio relativo**:

$$y_t=\frac{C_{t+h}}{C_t},\qquad \hat C_{t+h}=\hat y_t\,C_t.$$

El cambio relativo de 2023 se parece a los cambios relativos de 2015, aunque los precios sean otros. Las entradas también se dividen por el costo de hoy.

In [ ]:
relativas = datos[variables].div(datos["hoy"], axis=0).drop(columns="hoy")
relativas["mes_sen"] = np.sin(2 * np.pi * datos.index.month / 12)
relativas["mes_cos"] = np.cos(2 * np.pi * datos.index.month / 12)
razon = datos["real"] / datos["hoy"]

errores_cambio, pron_cambio = competir(relativas, razon, lambda pred, tramo: pred * datos["hoy"][tramo].values)

todos = pd.concat([bases,
                   errores_nivel.rename(index=lambda n: n + " (nivel)"),
                   errores_cambio.rename(index=lambda n: n + " (cambio)")])
todos.sort_values("validación").plot.barh(figsize=(9, 5), color=["#16302B", "#E0A526"])
plt.xlabel("MAPE (%)"); plt.title("Doce candidatos: validación para escoger, prueba para reportar"); plt.gca().invert_yaxis(); plt.show()
display(todos.sort_values("validación").round(1))

### 3.4 El veredicto

La regla quedó fijada antes de mirar: gana quien tenga menor MAPE en **validación**, y su error en **prueba** es el que se le reporta al cliente.

In [ ]:
mejor = todos["validación"].idxmin()
resumen = todos["prueba"].round(1)
print(f"Ganador en validación: {mejor}")
print(f"Su MAPE en prueba: {resumen[mejor]} %   |   MAPE del ingenuo en prueba: {resumen['ingenuo']} %")

pronosticos_prueba = pd.concat([tabla.loc[datos.index[P], ["ingenuo", "estacional"]],
                                pron_nivel.rename(columns=lambda n: n + " (nivel)"),
                                pron_cambio.rename(columns=lambda n: n + " (cambio)")], axis=1)
plt.figure(figsize=(11, 3.8))
plt.plot(fechas, datos["real"][P], color="black", lw=2.5, label="real")
plt.plot(fechas, pronosticos_prueba["ingenuo"], color="#E0A526", label="ingenuo")
if mejor != "ingenuo":
    plt.plot(fechas, pronosticos_prueba[mejor], color="#A8432C", ls="--", label=mejor)
plt.title("El ganador contra la línea base"); plt.ylabel("pesos"); plt.legend(); plt.show()

Lean el resultado con calma. Puede pasar que el ganador de validación pierda en prueba contra el pronóstico ingenuo. Eso no es un error del procedimiento: es el procedimiento diciendo la verdad sobre un dato con poca memoria en sus variaciones, como vimos en la sección 2.1. Un modelo que empata con la línea base solo agrega complejidad, y la respuesta honesta para el cliente es la más simple que no pierde.

### 3.5 Explicar: ¿qué ingrediente mueve el costo?

Como el costo es una suma, el alza total se reparte de forma exacta entre ingredientes:

$$C_T-C_1=\sum_{i=1}^{5}q_i\,(p_{i,T}-p_{i,1}).$$

In [ ]:
participacion = (costo_por_ingrediente.iloc[-1] / olla.iloc[-1] * 100).round(1).sort_values(ascending=False)
print("Participación en el costo, diciembre de 2024 (%):")
print(participacion)

alza = costo_por_ingrediente.iloc[-1] - costo_por_ingrediente.iloc[0]
print("\nDe cada 100 pesos que subió la olla entre 2013 y 2024, cuántos puso cada ingrediente:")
print((alza / alza.sum() * 100).round(1).sort_values(ascending=False))

costo_por_ingrediente.plot.area(lw=0, cmap="YlOrBr_r")
plt.title("Costo de la olla por ingrediente"); plt.xlabel(""); plt.ylabel("pesos"); plt.legend(fontsize=8); plt.show()

Esta explicación es una descomposición contable: dice cuánto pesa cada ingrediente, no por qué subió su precio. Para el porqué harían falta otros datos, como el precio del maíz que come el cerdo o el clima de la cosecha.

## 4. Comunicación

### 4.1 Decidir: ¿con cuánto se presupuesta la olla?

Un pronóstico de un solo número deja al cliente sin saber cuánto puede fallar. Usamos los errores de la prueba para ponerle un margen. Sea $u_t=C_{t+h}/\hat C_{t+h}-1$ lo que el costo real superó al pronóstico. El margen es el cuantil 0,90 de esos valores, y el costo para presupuestar es

$$\hat C_{T+h}\,\bigl(1+\hat Q_{0{,}90}(u)\bigr).$$

Escogemos el método más simple entre los que no pierden: si el ganador de validación no mejora al ingenuo en la prueba, nos quedamos con el ingenuo.

In [ ]:
elegido = mejor if resumen[mejor] < resumen["ingenuo"] else "ingenuo"
u = datos["real"][P] / pronosticos_prueba[elegido] - 1
margen = u.quantile(0.90)

hoy = olla.index[-1]
ultima = X.iloc[[-1]]
if elegido == "ingenuo":
    punto = olla.iloc[-1]
elif elegido == "estacional":
    punto = olla.shift(12 - H).iloc[-1]
else:
    nombre, tipo = elegido.rsplit(" (", 1)
    modelo = cinco_modelos()[nombre]
    if tipo.startswith("nivel"):
        punto = float(modelo.fit(datos[variables], datos["real"]).predict(ultima[variables])[0])
    else:
        ultima_rel = ultima[variables].div(ultima["hoy"], axis=0).drop(columns="hoy")
        ultima_rel["mes_sen"] = np.sin(2 * np.pi * hoy.month / 12); ultima_rel["mes_cos"] = np.cos(2 * np.pi * hoy.month / 12)
        punto = float(modelo.fit(relativas, razon).predict(ultima_rel)[0]) * olla.iloc[-1]

meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
destino_mes = hoy + pd.DateOffset(months=H)
print(f"Método elegido: {elegido} (MAPE en prueba {resumen[elegido]} %)")
print(f"Pronóstico para {meses[destino_mes.month - 1]} de {destino_mes.year}: ${punto:,.0f}")
print(f"En 9 de cada 10 meses de prueba, el costo real no superó al pronóstico en más de {margen * 100:.1f} %")
print(f"Costo para presupuestar: ${punto * (1 + margen):,.0f}")

### La decisión en tres frases

Escríbanla para alguien que no sabe qué es una regresión. Una frase para lo que recomiendan, una para la evidencia y una para lo que puede salir mal.

1. **Recomendación:**
2. **Evidencia:**
3. **Riesgo:**

### 4.2 Bitácora de decisiones

| Decisión | Alternativa descartada | Por qué |
|---|---|---|
| Variedades de la receta (papa capira, yuca ICA, plátano dominico hartón) | Otras variedades | Se reportan todos los meses en la Galería |
| Horizonte de tres meses | Un mes, seis meses | Es lo que dura la carta impresa |
| MAPE como métrica | Error en pesos | El cliente piensa en márgenes |
| Tres tramos temporales: hasta 2019, 2020 a 2021 y 2022 a 2024 | Separación al azar; escoger mirando la prueba | Al barajar, el modelo ve el futuro; si se escoge en la prueba, la prueba deja de ser examen |
| Pronosticar el cambio relativo además del nivel | Solo el nivel | Árboles y vecinos no extrapolan por encima de lo visto |
| Margen con el percentil 90 de los meses de prueba | Intervalo teórico | Sale de errores observados y se explica sin fórmulas |

### 4.3 Riesgos y mantenimiento

Respondan en dos o tres líneas cada una.

- El dato llega hasta diciembre de 2024. ¿Qué habría que hacer para usar esto hoy?
- El precio es mayorista. ¿En qué se equivoca el cliente si lo lee como el precio de la tienda?
- ¿Cada cuánto habría que revisar si el pronóstico ingenuo sigue siendo el mejor?

### 4.4 Procedencia

Lo último que hace un cuaderno serio es decir de dónde salió su dato.

In [ ]:
procedencia()